# Broadcasting

**Goal:** Understand PyTorch's broadcasting rules by manually expanding tensors to a common shape, validating against auto-broadcast, and exploring valid vs invalid shape pairs.

## Configuration

Device, random seed, and default dtype come from `shared.config`, which reads `config.toml`.  
On Apple Silicon this resolves to `mps`; on CUDA machines it resolves to `cuda`; otherwise `cpu`.

In [1]:
import sys
from pathlib import Path

import torch


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure

device = configure()  # reads config.toml -> device/seed/dtype (mps on Apple Silicon)
print("running on:", device)

running on: mps


## Broadcasting Rules

Broadcasting aligns shapes from the **right**, padding missing leading dimensions with 1.  
Two dimensions are compatible when they are **equal** or **one of them is 1**.  
The output shape is the maximum along each aligned dimension.

In [2]:
def broadcast_shapes_manual(shape_a: tuple, shape_b: tuple) -> tuple:
    """Compute broadcast output shape following NumPy/PyTorch rules.

    Raises:
        ValueError: If shapes are incompatible.
    """
    # Pad the shorter shape on the left with 1s
    ndim = max(len(shape_a), len(shape_b))
    a = (1,) * (ndim - len(shape_a)) + tuple(shape_a)
    b = (1,) * (ndim - len(shape_b)) + tuple(shape_b)

    out = []
    for da, db in zip(a, b):
        if da == db:
            out.append(da)
        elif da == 1:
            out.append(db)
        elif db == 1:
            out.append(da)
        else:
            raise ValueError(f"Shapes {shape_a} and {shape_b} are incompatible at dims {da} vs {db}")
    return tuple(out)


# Example from README: (4, 1, 7) + (1, 3, 7) -> (4, 3, 7)
result = broadcast_shapes_manual((4, 1, 7), (1, 3, 7))
print("Manual broadcast shape:", result)
assert result == (4, 3, 7)

# Bias addition: (32, 128) + (128,) -> (32, 128)
result2 = broadcast_shapes_manual((32, 128), (128,))
print("Bias broadcast shape:", result2)
assert result2 == (32, 128)
print("Shape inference correct ✓")

Manual broadcast shape: (4, 3, 7)
Bias broadcast shape: (32, 128)
Shape inference correct ✓


## Manual Expand vs Auto-Broadcast

In [3]:
# Bias addition: activation (batch, hidden) + bias (hidden,)
batch, hidden = 4, 8
activation = torch.randn(batch, hidden, device=device)
bias = torch.randn(hidden, device=device)

# Manual: explicitly expand bias to (batch, hidden) — no data copy
bias_expanded = bias.unsqueeze(0).expand(batch, hidden)  # shape: (4, 8)
result_manual = activation + bias_expanded

# Auto-broadcast
result_auto = activation + bias

print("activation shape:", activation.shape)
print("bias shape:      ", bias.shape)
print("bias_expanded:   ", bias_expanded.shape)
print("result shape:    ", result_auto.shape)

assert torch.allclose(result_manual, result_auto, atol=1e-6), "Manual expand != auto-broadcast!"
print("Manual expand matches auto-broadcast ✓")

# Show expand does NOT copy data
print("\nbias data_ptr:", bias.data_ptr())
print("bias_expanded data_ptr:", bias_expanded.data_ptr(), " (same storage, no copy)")

activation shape: torch.Size([4, 8])
bias shape:       torch.Size([8])
bias_expanded:    torch.Size([4, 8])
result shape:     torch.Size([4, 8])
Manual expand matches auto-broadcast ✓

bias data_ptr: 48708311616
bias_expanded data_ptr: 48708311616  (same storage, no copy)


## Rank-3 Broadcasting: `(4, 1, 7) + (1, 3, 7)`

In [4]:
A = torch.randn(4, 1, 7, device=device)
B = torch.randn(1, 3, 7, device=device)

# Manual: expand both to (4, 3, 7)
A_exp = A.expand(4, 3, 7)  # no copy
B_exp = B.expand(4, 3, 7)
result_manual3 = A_exp + B_exp

# Auto-broadcast
result_auto3 = A + B

print("A shape:", A.shape, "B shape:", B.shape, "-> result:", result_auto3.shape)
assert torch.allclose(result_manual3, result_auto3, atol=1e-6), "3D broadcast mismatch!"
print("3D broadcast: manual expand == auto ✓")

# Verify indexing: C[i, j, k] = A[i, 0, k] + B[0, j, k]
i, j, k = 2, 1, 5
assert torch.isclose(result_auto3[i, j, k], A[i, 0, k] + B[0, j, k]), "Indexing check failed!"
print(f"C[{i},{j},{k}] = A[{i},0,{k}] + B[0,{j},{k}]  ✓")

A shape: torch.Size([4, 1, 7]) B shape: torch.Size([1, 3, 7]) -> result: torch.Size([4, 3, 7])


3D broadcast: manual expand == auto ✓


C[2,1,5] = A[2,0,5] + B[0,1,5]  ✓


## Reshape to Broadcast: Per-Example Scaling

In [5]:
# Per-example scale factor: (batch, 1) * (batch, features) -> (batch, features)
batch, features = 6, 10
X = torch.randn(batch, features, device=device)
scale = torch.rand(batch, device=device)  # one scale per example

# Manual: reshape scale to (batch, 1) so it broadcasts over features
scale_col = scale.reshape(batch, 1)  # (6, 1)
result_reshape = scale_col * X

# Auto: (batch,) cannot directly broadcast with (batch, features) — need unsqueeze or reshape
result_auto_scaled = scale.unsqueeze(1) * X

assert torch.allclose(result_reshape, result_auto_scaled, atol=1e-6)
print("Per-example scaling:", scale_col.shape, "*", X.shape, "->", result_reshape.shape, "✓")

Per-example scaling: torch.Size([6, 1]) * torch.Size([6, 10]) -> torch.Size([6, 10]) ✓


## Valid vs Invalid Shape Pairs

In [6]:
import torch

test_cases = [
    # (shape_a, shape_b, valid)
    ((3, 1), (3, 4), True),
    ((5, 4), (4,),   True),
    ((1,),   (8, 8), True),
    ((3, 4), (3, 5), False),  # 4 != 5 and neither is 1
    ((2, 3), (5, 3), False),  # 2 != 5 and neither is 1
]

for shape_a, shape_b, should_work in test_cases:
    a = torch.randn(*shape_a, device=device)
    b = torch.randn(*shape_b, device=device)
    try:
        result = a + b
        status = f"OK -> {tuple(result.shape)}"
        worked = True
    except RuntimeError as e:
        status = f"ERROR: {e}"
        worked = False

    marker = "✓" if worked == should_work else "✗ UNEXPECTED"
    print(f"{str(shape_a):>12} + {str(shape_b):<12} {'valid' if should_work else 'invalid':>7}  {marker}  {status}")

      (3, 1) + (3, 4)         valid  ✓  OK -> (3, 4)
      (5, 4) + (4,)           valid  ✓  OK -> (5, 4)
        (1,) + (8, 8)         valid  ✓  OK -> (8, 8)
      (3, 4) + (3, 5)       invalid  ✓  ERROR: The size of tensor a (4) must match the size of tensor b (5) at non-singleton dimension 1
      (2, 3) + (5, 3)       invalid  ✓  ERROR: The size of tensor a (2) must match the size of tensor b (5) at non-singleton dimension 0


## Idiomatic PyTorch & Gradient Reduction

During backprop, gradients through a broadcasted axis must be **summed** over every expanded dimension — because each element of the smaller tensor contributed to multiple outputs.

In [7]:
# Gradient reduction for bias addition
n, d = 8, 4
X_g = torch.randn(n, d, device=device, requires_grad=False)
b_g = torch.randn(d, device=device, requires_grad=True)

Y = X_g + b_g           # auto-broadcast: b_g treated as (1, d) -> (n, d)
G = torch.ones_like(Y)  # upstream gradient

Y.backward(G)

# dL/db = sum over rows of G (because b influenced n output rows)
db_manual = G.sum(dim=0)
assert torch.allclose(b_g.grad, db_manual, atol=1e-6), "Gradient reduction mismatch!"
print("b_g.grad shape:", b_g.grad.shape)
print("Expected sum over batch:", db_manual)
print("Gradient reduction correct ✓")

b_g.grad shape: torch.Size([4])
Expected sum over batch: tensor([8., 8., 8., 8.], device='mps:0')
Gradient reduction correct ✓


## Takeaways

- **Rule:** align shapes from the right; compatible dims are equal or one is 1; missing dims act as 1.
- **`expand` vs `repeat`:** `expand` is a zero-copy view (same memory, stride trick); `repeat` copies data.
- **Reshape/unsqueeze matters:** `(batch,)` cannot broadcast against `(batch, features)` — you need `(batch, 1)`.
- **Backward pass:** gradients sum over every broadcasted axis. This is why `dL/db = G.sum(dim=0)` in a linear layer.
- **Silent bugs:** compatible shapes don't guarantee correct semantics — a `(batch, 1)` and `(features,)` will combine to `(batch, features)`, which may be exactly wrong.